## Workflow guide
This analysis aggregates patient utilization by sex and saves a cohort-demographics table. It intentionally uses only aggregated results suitable for a portfolio dashboard.

In [0]:
from pyspark.sql import functions as F

CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = "clinical_portfolio"
BASE = f"{CATALOG}.{SCHEMA}"

patient_utilization = spark.table(f"{BASE}.gold_patient_utilization")
condition_summary = spark.table(f"{BASE}.gold_condition_summary")

display(patient_utilization)

person_id,year_of_birth,gender_source_value,observation_period_start_date,observation_period_end_date,visit_count,condition_count,observation_days
7419654189246804363,1978,F,2025-01-04,2025-03-17,2,3,72
4905020018283288165,1986,M,2025-02-10,2025-05-22,2,2,101
7357811929381004785,1994,F,2025-02-28,2025-02-28,1,1,0
3215699940703695032,1969,M,2025-04-06,2025-04-08,1,1,2
3871179191872680772,2001,F,2025-04-19,2025-04-19,1,1,0
2763232168323479912,1982,M,2025-05-03,2025-05-03,1,1,0


In [0]:
cohort_demographics = (
    patient_utilization
    .groupBy("gender_source_value")
    .agg(
        F.countDistinct("person_id").alias("patient_count"),
        F.round(F.avg("visit_count"), 2).alias("avg_visits_per_patient"),
        F.round(F.avg("condition_count"), 2).alias("avg_conditions_per_patient"),
        F.round(F.avg("observation_days"), 2).alias("avg_observation_days")
    )
    .orderBy("gender_source_value")
)

display(cohort_demographics)

gender_source_value,patient_count,avg_visits_per_patient,avg_conditions_per_patient,avg_observation_days
F,3,1.33,1.67,24.0
M,3,1.33,1.33,34.33


### Interpret the cohort safely
These values are aggregated by sex and are used to compare utilization patterns. The synthetic dataset is deliberately small, so the visual is a pipeline demonstration rather than a clinical conclusion.

In [0]:
display(
    condition_summary
    .select(
        "condition_source_value",
        "condition_event_count",
        "unique_patient_count",
        "first_recorded_date",
        "last_recorded_date"
    )
    .orderBy(F.desc("condition_event_count"))
)

condition_source_value,condition_event_count,unique_patient_count,first_recorded_date,last_recorded_date
HTN,4,3,2025-01-04,2025-05-22
T2D,2,2,2025-03-15,2025-05-03
ASTHMA,2,2,2025-02-10,2025-04-19
MIGRAINE,1,1,2025-02-28,2025-02-28


In [0]:
cohort_demographics.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{BASE}.gold_cohort_demographics")

display(spark.table(f"{BASE}.gold_cohort_demographics"))

gender_source_value,patient_count,avg_visits_per_patient,avg_conditions_per_patient,avg_observation_days
F,3,1.33,1.67,24.0
M,3,1.33,1.33,34.33


### Persist the cohort data product
Saving the aggregate as a Delta table allows Power BI exports and future analyses to consume a stable Gold output rather than repeating the cohort calculation.

# Cohort analysis

This notebook aggregates the curated patient-level data into cohort demographics. It compares patient counts, visits, conditions and observation time by sex while keeping the analysis at an aggregated, portfolio-safe level.